# MedFocus walkthrough

End-to-end run on a single MedGround-Bench sample. Demonstrates:
1. Loading an LVLM via `load_lvlm`.
2. Building a reference-CXR pool and a MedSAM client.
3. Calling `MedFocus.attribute(image, question)` and inspecting outputs.
4. Visualizing the winning concept bbox and per-token Δ for the answer span.

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.environ.setdefault('MEDFOCUS_DATA_ROOT', '/PATH/TO/physionet.org/files')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from medfocus import MedFocus, load_lvlm, load_medground
from medfocus.config import load_config
from medfocus.data.io import safe_open_image, resize_pad
from medfocus.data.imagenome import load_imagenome
from medfocus.data.medground_bench import attach_source_metadata
from medfocus.medsam import MedSAMClient
from medfocus.ot.reference import ReferenceCXRPool
from medfocus.attribution.visualization import overlay_boxes

## 1. Load configuration and the released benchmark

In [ ]:
cfg = load_config('../configs')
direct = load_medground('../data/medground_bench', split='direct')
print(f'{len(direct):,} direct-mode samples')

# Pick one record. Filter to a model and an interesting dataset.
record = next(r for r in direct if r['model'] == 'medgemma1_5_4b' and r['dataset'] == 'imagenome')
print(record)

## 2. Recover image / question / GT boxes from the source dataset

In [ ]:
imagenome = load_imagenome(cfg.datasets.data_root, suffixes=cfg.datasets.question_suffixes)
joined = attach_source_metadata([record], {'imagenome': imagenome})[0]
img = resize_pad(safe_open_image(joined['imgpath']), cfg.datasets.image['resize_width'])
question = joined['question_direct'] or joined['question']
answer = joined['prediction_direct']
print('attribute:', joined.get('attribute'))
print('question:', question)
print('GT boxes :', joined['locations'])
img

## 3. Construct the MedFocus pipeline

In [ ]:
adapter = load_lvlm('medgemma1_5_4b')
medsam  = MedSAMClient(cfg.medfocus.medsam.model_id)
ref_pool = ReferenceCXRPool.from_directory(
    images_dir=cfg.reference_pool.images_dir,
    masks_dir=cfg.reference_pool.masks_dir,
    concepts=cfg.medfocus.concepts,
    candidates=cfg.reference_pool.candidates or None,
    selection_grid=cfg.medfocus.ot.selection_grid,
    epsilon=cfg.medfocus.ot.epsilon,
    lambda_marginal=cfg.medfocus.ot.lambda_marginal,
)
mf = MedFocus(
    adapter, ref_pool=ref_pool, medsam=medsam,
    concepts=cfg.medfocus.concepts,
    composites=cfg.medfocus.composites,
    tau=cfg.medfocus.intervention.tau,
    image_size=cfg.datasets.image['resize_width'],
    transfer_grid=cfg.medfocus.ot.transfer_grid,
    epsilon=cfg.medfocus.ot.epsilon,
    lambda_marginal=cfg.medfocus.ot.lambda_marginal,
    mass_quantile=cfg.medfocus.ot.mass_quantile,
    intervention_baseline=cfg.medfocus.intervention.baseline,
)

## 4. Run attribution

In [ ]:
result = mf.attribute(img, question, precomputed_answer=answer)
print('answer       :', result.answer)
print('concept      :', result.concept)
print('bbox (xyxy)  :', result.bbox)
print('Δ            :', result.delta)
print('fallback     :', result.fallback_used)
print('reference    :', result.reference_path)

## 5. Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(overlay_boxes(img, joined['locations'], color='red', label='GT'))
axes[0].set_title('Ground truth')
axes[0].axis('off')
axes[1].imshow(overlay_boxes(img, [result.bbox], color='yellow', label=result.concept))
axes[1].set_title(f'MedFocus: {result.concept}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
names = list(result.deltas.keys())
deltas = [result.deltas[n] for n in names]
order = np.argsort(deltas)[::-1]
names = [names[i] for i in order]
deltas = [deltas[i] for i in order]
plt.figure(figsize=(8, 4))
plt.barh(range(len(names)), deltas)
plt.yticks(range(len(names)), names)
plt.gca().invert_yaxis()
plt.xlabel('Δ (cumulative log-prob drop)')
plt.title('Concept influence on the answer')
plt.tight_layout()
plt.show()